# Fundamentals of Computer Science using Python – II
## Complete Notes: Unit 4 (Regression) & Unit 5 (Classification)
---

# UNIT 4 — REGRESSION: MODEL TRAINING AND EVALUATION

---

## 4.0 What is Supervised Learning? (Foundation)

Before regression, you need to understand the broader picture.

**Supervised Learning** means we train a model using **labeled data** — data where we already know the correct answers (called **labels** or **targets**).

```
Input (Features X)  →  [Model]  →  Output (Target y)
Area, Bedrooms      →  [Model]  →  House Price
```

The model learns a **mapping function** `f` such that:

```
y = f(X) + ε
```

Where:
- `y` = target (what we want to predict)
- `X` = features (inputs)
- `f` = the function the model learns
- `ε` (epsilon) = irreducible error (noise in real-world data)

**Two types of supervised learning:**

| Type | Output | Example |
|------|--------|---------|
| **Regression** | Continuous number | House price, temperature, salary |
| **Classification** | Discrete category | Spam/not spam, disease diagnosis |

---

## 4.1 Simple Linear Regression — Deep Dive

### 4.1.1 The Intuition

Imagine you have data points scattered on a graph. Simple Linear Regression finds the **single best straight line** that passes through those points — the line that is, on average, as close to all points as possible.

```
y
|          *
|       *    *
|    *
|  *
|*
+-------------------> x
```

The line we find: `y = a + bx`

### 4.1.2 The Mathematics

The equation of the regression line is:

```
ŷ = β₀ + β₁x
```

Where:
- `ŷ` (y-hat) = predicted value
- `β₀` (beta-zero) = **intercept** — value of y when x = 0
- `β₁` (beta-one) = **slope/coefficient** — how much y changes when x increases by 1

**How are β₀ and β₁ calculated?**

We use the **Ordinary Least Squares (OLS)** method — find the line that minimizes the sum of squared differences between actual and predicted values.

The **cost function** (what we minimize):

```
SSE = Σᵢ (yᵢ - ŷᵢ)²  =  Σᵢ (yᵢ - β₀ - β₁xᵢ)²
```

SSE = Sum of Squared Errors

Taking the derivative with respect to β₀ and β₁ and setting to zero gives the **closed-form solution**:

```
β₁ = [n·Σ(xᵢyᵢ) - Σxᵢ·Σyᵢ] / [n·Σ(xᵢ²) - (Σxᵢ)²]

β₀ = ȳ - β₁·x̄
```

Where:
- `n` = number of data points
- `x̄` = mean of x values
- `ȳ` = mean of y values

Equivalently (and more elegantly):

```
β₁ = Σ[(xᵢ - x̄)(yᵢ - ȳ)] / Σ[(xᵢ - x̄)²]
   = Cov(X, Y) / Var(X)
```

### 4.1.3 Worked Example (by hand)

Given data: x = [1, 2, 3, 4, 5], y = [2, 4, 5, 4, 5]

Step 1: Calculate means
```
x̄ = (1+2+3+4+5)/5 = 3
ȳ = (2+4+5+4+5)/5 = 4
```

Step 2: Calculate β₁
```
Numerator:   Σ(xᵢ-x̄)(yᵢ-ȳ)
  = (1-3)(2-4) + (2-3)(4-4) + (3-3)(5-4) + (4-3)(4-4) + (5-3)(5-4)
  = (-2)(-2) + (-1)(0) + (0)(1) + (1)(0) + (2)(1)
  = 4 + 0 + 0 + 0 + 2 = 6

Denominator: Σ(xᵢ-x̄)²
  = 4 + 1 + 0 + 1 + 4 = 10

β₁ = 6/10 = 0.6
```

Step 3: Calculate β₀
```
β₀ = ȳ - β₁·x̄ = 4 - 0.6×3 = 4 - 1.8 = 2.2
```

Regression line: **ŷ = 2.2 + 0.6x**

Prediction for x=6: ŷ = 2.2 + 0.6(6) = **5.8**

### 4.1.4 Assumptions of Linear Regression

Linear regression only gives valid results if these assumptions hold:

1. **Linearity** — The relationship between X and y is actually linear
2. **Independence** — Each observation is independent of others
3. **Homoscedasticity** — The variance of errors is constant across all values of X
4. **Normality of errors** — The residuals (errors) follow a normal distribution
5. **No multicollinearity** (for multiple regression) — Features are not highly correlated with each other

> **Real-world tip:** In practice, these assumptions are often slightly violated. Linear regression is still used and often works well. The key is to know *when* violations become serious enough to switch models.

### 4.1.5 sklearn Implementation

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# --- Load and prepare data ---
df = pd.read_csv("student_scores.csv")
print(df.head())
print(df.describe())
print(df.isnull().sum())  # check for missing values

# --- Feature and target ---
X = df[['Hours']]   # must be 2D (double brackets)
y = df['Scores']    # 1D is fine for target

# --- Split data ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

# --- Train model ---
model = LinearRegression()
model.fit(X_train, y_train)

# --- Coefficients ---
print(f"Intercept (β₀): {model.intercept_:.4f}")
print(f"Coefficient (β₁): {model.coef_[0]:.4f}")
print(f"Equation: y = {model.intercept_:.2f} + {model.coef_[0]:.2f}x")

# --- Predict ---
y_pred = model.predict(X_test)

# --- Evaluate ---
r2   = r2_score(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)

print(f"\nR² Score   : {r2:.4f}")
print(f"MSE        : {mse:.4f}")
print(f"RMSE       : {rmse:.4f}")
print(f"MAE        : {mae:.4f}")

# --- Visualize ---
plt.scatter(X_test, y_test, color='blue', label='Actual', alpha=0.7)
plt.plot(X_test, y_pred, color='red', linewidth=2, label='Predicted')
plt.xlabel('Hours Studied')
plt.ylabel('Score')
plt.title('Simple Linear Regression')
plt.legend()
plt.show()
```

---

## 4.2 Multiple Linear Regression

### 4.2.1 Theory

When there are multiple independent variables (features):

```
ŷ = β₀ + β₁x₁ + β₂x₂ + β₃x₃ + ... + βₙxₙ
```

In **matrix notation** (how computers actually compute it):

```
ŷ = Xβ
```

Where:
- `X` is the feature matrix (n_samples × n_features)
- `β` is the coefficient vector

The OLS solution in matrix form:

```
β = (XᵀX)⁻¹Xᵀy
```

This is called the **Normal Equation**. sklearn uses this (or more numerically stable variants like SVD decomposition) internally.

### 4.2.2 Interpretation of Coefficients

Each coefficient `βᵢ` means:

> "If feature `xᵢ` increases by 1 unit, and all other features remain constant, then `y` changes by `βᵢ` units."

This is called **ceteris paribus** (all else equal) interpretation.

Example: If a model predicts house price and β₁ (for area) = 500:
> "Each additional square foot adds ₹500 to the predicted price, assuming bedrooms and age stay the same."

### 4.2.3 Categorical Variables — One-Hot Encoding

Linear Regression needs numbers. Categorical features must be converted.

**Label Encoding** (wrong for nominal categories):
```python
# Maps: Male=0, Female=1
# Problem: implies Female > Male mathematically!
```

**One-Hot Encoding** (correct):
```python
import pandas as pd

df = pd.get_dummies(df, columns=['Fuel_Type', 'Transmission'], drop_first=True)
# drop_first=True avoids multicollinearity (dummy variable trap)
```

**The Dummy Variable Trap:** If you have k categories and create k dummy columns, one is always predictable from the others. Always use k-1 columns (`drop_first=True`).

### 4.2.4 sklearn Implementation

```python
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder

# Load data
df = pd.read_csv("Housing.csv")

# Check and handle missing values
print(df.isnull().sum())
df = df.dropna()  # or df.fillna(df.mean())

# Encode categorical columns
df = pd.get_dummies(df, drop_first=True)

# Features and target
X = df.drop('price', axis=1)
y = df['price']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Train
model = LinearRegression()
model.fit(X_train, y_train)

# Results
y_pred = model.predict(X_test)
print(f"R²   : {r2_score(y_test, y_pred):.4f}")
print(f"MSE  : {mean_squared_error(y_test, y_pred):.2f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")

# Coefficient table
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
}).sort_values('Coefficient', ascending=False)
print(coef_df)
```

---

## 4.3 Polynomial Regression

### 4.3.1 Why Polynomial?

Linear regression assumes a straight-line relationship. When data looks curved, a straight line will systematically underfit.

```
Linear fit on curved data:
y
|       * * *
|     *       *
|   *     ----  ← this line misses the curve
| *---
|*
+-------------------> x

Polynomial fit:
y
|       * * *
|     *     *
|   *         ← the curve follows the data
| *
|*
+-------------------> x
```

### 4.3.2 The Mathematics

A degree-d polynomial regression:

```
ŷ = β₀ + β₁x + β₂x² + β₃x³ + ... + βdx^d
```

This is still **linear regression internally** — it's linear in the coefficients β. We just create new features x², x³, etc.

The transformation step:

```
Original:    [x]
After poly:  [1, x, x², x³]  (degree=3, include_bias=True)
```

### 4.3.3 The Bias-Variance Tradeoff (Critical Concept)

This is one of the most important concepts in all of machine learning.

```
Total Error = Bias² + Variance + Irreducible Noise
```

**Bias** = how far off your predictions are on average (systematic error)
**Variance** = how much your predictions change with different training data (sensitivity)

| Degree | Bias | Variance | Problem |
|--------|------|----------|---------|
| Too low (d=1) | High | Low | **Underfitting** — model too simple |
| Just right | Medium | Medium | **Good generalization** |
| Too high (d=10+) | Low | High | **Overfitting** — model memorizes training data |

**Overfitting:** Model performs great on training data but poorly on new (test) data.
**Underfitting:** Model performs poorly on both training and test data.

How to detect:
```python
# Compare training vs test R²
r2_train = r2_score(y_train, model.predict(X_train_poly))
r2_test  = r2_score(y_test, model.predict(X_test_poly))

# Overfitting: r2_train >> r2_test (large gap)
# Underfitting: both r2_train and r2_test are low
print(f"Train R²: {r2_train:.4f}")
print(f"Test  R²: {r2_test:.4f}")
```

### 4.3.4 sklearn Implementation

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score

# Sample data
X = np.array([5, 15, 25, 35, 45, 55]).reshape(-1, 1)
y = np.array([5, 20, 14, 32, 22, 38])

# Method 1: Manual transformation
poly = PolynomialFeatures(degree=2, include_bias=True)
X_poly = poly.fit_transform(X)
print("Original shape:", X.shape)
print("Polynomial shape:", X_poly.shape)  # (6, 3) = [1, x, x²]

model = LinearRegression()
model.fit(X_poly, y)
y_pred = model.predict(X_poly)
print(f"R² (degree=2): {r2_score(y, y_pred):.4f}")
print(f"Coefficients: {model.coef_}")
print(f"Intercept: {model.intercept_:.4f}")

# Method 2: Pipeline (cleaner — industry standard)
pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=3)),
    ('model', LinearRegression())
])
pipe.fit(X, y)
y_pred_pipe = pipe.predict(X)
print(f"\nR² (degree=3, pipeline): {r2_score(y, y_pred_pipe):.4f}")

# Predict for new values
X_new = np.arange(5).reshape(-1, 1)
print("Predictions for x=0,1,2,3,4:", pipe.predict(X_new))

# Compare multiple degrees
plt.figure(figsize=(12, 4))
for i, degree in enumerate([1, 2, 3, 5], 1):
    pipe_d = Pipeline([('poly', PolynomialFeatures(degree=degree)), ('lr', LinearRegression())])
    pipe_d.fit(X, y)
    X_plot = np.linspace(X.min(), X.max(), 200).reshape(-1, 1)
    plt.subplot(1, 4, i)
    plt.scatter(X, y, color='black', s=50)
    plt.plot(X_plot, pipe_d.predict(X_plot), color='red')
    plt.title(f'Degree {degree}\nR²={r2_score(y, pipe_d.predict(X)):.3f}')
plt.tight_layout()
plt.show()
```

---

## 4.4 Evaluation Metrics — Complete Theory

### 4.4.1 R² — Coefficient of Determination

**Formula:**
```
R² = 1 - SS_res / SS_tot

SS_res = Σ(yᵢ - ŷᵢ)²   ← residual sum of squares
SS_tot = Σ(yᵢ - ȳ)²    ← total sum of squares
```

**Interpretation:**
- R² = 1.0 → Perfect fit (model explains 100% of variance)
- R² = 0.85 → Model explains 85% of the variance in y
- R² = 0.0 → Model is no better than predicting the mean
- R² < 0 → Model is WORSE than predicting the mean (severely overfit or wrong model)

**Important:** R² alone is not enough. A high R² on training data with low R² on test data = overfitting.

**Adjusted R²** (better for multiple regression):
```
Adj R² = 1 - (1-R²)(n-1)/(n-k-1)
```
Where n = samples, k = number of features. Penalizes adding useless features.

### 4.4.2 MSE — Mean Squared Error

```
MSE = (1/n) Σ(yᵢ - ŷᵢ)²
```

- Units: squared units of y (if y is in ₹, MSE is in ₹²)
- Penalizes large errors more heavily (because of squaring)
- Always ≥ 0; lower is better

### 4.4.3 RMSE — Root Mean Squared Error

```
RMSE = √MSE = √[(1/n) Σ(yᵢ - ŷᵢ)²]
```

- Same units as y — much more interpretable than MSE
- If RMSE = 5000 and y is house price in ₹, your predictions are off by ~₹5000 on average

### 4.4.4 MAE — Mean Absolute Error

```
MAE = (1/n) Σ|yᵢ - ŷᵢ|
```

- More robust to outliers than MSE/RMSE (doesn't square the errors)
- If one prediction is wildly wrong, MSE gets more affected than MAE

**When to use which:**
- Use **RMSE** when large errors are especially bad (you want to penalize them)
- Use **MAE** when outliers exist and you don't want them to dominate the metric
- Use **R²** for a standardized, percentage-based measure of fit

```python
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np

r2   = r2_score(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae  = mean_absolute_error(y_test, y_pred)

print(f"R²   : {r2:.4f}  (proportion of variance explained)")
print(f"MSE  : {mse:.4f}  (average squared error)")
print(f"RMSE : {rmse:.4f}  (average error in original units)")
print(f"MAE  : {mae:.4f}  (average absolute error, robust to outliers)")
```

---

## 4.5 Train-Test Split — The Why and How

### Why Split Data?

If you train and evaluate on the same data, the model will score perfectly — but it has just **memorized** the data. This tells you nothing about how it performs on new, unseen data.

```
All Data (100%)
     |
     ├──── Training Set (70-80%) ─── Model learns from this
     |
     └──── Test Set (20-30%)    ─── Model is evaluated on this
                                    (model has NEVER seen this)
```

### The Leakage Problem

**Data Leakage** = when information from the test set accidentally influences the model or preprocessing. This gives falsely optimistic results.

**Example of leakage (WRONG):**
```python
# WRONG: Fit scaler on all data, then split
scaler.fit(X)           # learns stats from test data too!
X_scaled = scaler.transform(X)
X_train, X_test, ... = train_test_split(X_scaled, y)
```

**Correct approach:**
```python
# RIGHT: Split first, then fit scaler ONLY on train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
scaler.fit(X_train)            # learns stats ONLY from train
X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)  # apply same transform to test
```

### Parameters of train_test_split

```python
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% for testing (0.0 to 1.0)
    random_state=42,    # seed for reproducibility
    shuffle=True,       # shuffle before splitting (default True)
    stratify=y          # maintain class proportions (for classification)
)
```

**Output order:** `X_train, X_test, y_train, y_test` — memorize this!

### 🌟 Cross-Validation (Beyond Syllabus — Very Important)

A single split can be "lucky" or "unlucky". K-Fold Cross-Validation gives a more reliable estimate:

```
Data: [1][2][3][4][5]  (5 folds)

Fold 1: Train=[2,3,4,5]  Test=[1]
Fold 2: Train=[1,3,4,5]  Test=[2]
Fold 3: Train=[1,2,4,5]  Test=[3]
Fold 4: Train=[1,2,3,5]  Test=[4]
Fold 5: Train=[1,2,3,4]  Test=[5]

Final score = average of 5 test scores
```

```python
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model, X, y, cv=5, scoring='r2')
print(f"CV R² scores: {scores}")
print(f"Mean: {scores.mean():.4f} ± {scores.std():.4f}")
```

---

## 4.6 🌟 Feature Engineering (Extra — Industry Essential)

### Handling Missing Values

```python
# Strategy 1: Drop rows with missing values
df = df.dropna()

# Strategy 2: Fill with mean (numerical)
df['column'].fillna(df['column'].mean(), inplace=True)

# Strategy 3: Fill with median (better for skewed data)
df['column'].fillna(df['column'].median(), inplace=True)

# Strategy 4: Fill with mode (categorical)
df['column'].fillna(df['column'].mode()[0], inplace=True)

# sklearn imputer (use in pipeline to avoid leakage)
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')
X_train = imputer.fit_transform(X_train)
X_test  = imputer.transform(X_test)
```

### Feature Scaling

Tree-based models (Decision Tree, Random Forest) don't need scaling.
Distance-based and gradient-based models (Linear Regression, kNN, SVM, Neural Networks) DO need scaling.

```python
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# StandardScaler: z = (x - mean) / std → mean=0, std=1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# MinMaxScaler: z = (x - min) / (max - min) → range [0, 1]
scaler = MinMaxScaler()
```

### Binning / Discretization (from Practice Book Q.190–192)

```python
import pandas as pd
import numpy as np

# Example: categorize apartment prices
def categorize_price(price):
    if price > 3_000_000:
        return 'High'
    elif price < 2_000_000:
        return 'Low'
    else:
        return 'Medium'

df['price_category'] = df['price'].apply(categorize_price)

# Or using pd.cut (cleaner for multiple bins)
bins   = [0, 2_000_000, 3_000_000, float('inf')]
labels = ['Low', 'Medium', 'High']
df['price_category'] = pd.cut(df['price'], bins=bins, labels=labels)

# Age categorization (Practice Book Q.191)
bins   = [0, 30, 60, 120]
labels = ['Young', 'Middle-aged', 'Elderly']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)

# Income categorization (Practice Book Q.192)
bins   = [0, 30000, 70000, float('inf')]
labels = ['Low', 'Medium', 'High']
df['income_group'] = pd.cut(df['income'], bins=bins, labels=labels)
```

---

---

# UNIT 5 — CLASSIFICATION: MODEL TRAINING AND EVALUATION

---

## 5.0 Classification vs Regression — Key Difference

| | Regression | Classification |
|---|---|---|
| **Output** | Continuous number (e.g., 42500.0) | Discrete label (e.g., "spam", 0, 1) |
| **Examples** | Price, temperature, salary | Disease, spam, digit recognition |
| **Evaluation** | R², MSE, RMSE | Accuracy, Confusion Matrix, F1 |
| **Algorithms** | Linear/Polynomial Regression | kNN, Decision Tree, SVM, Random Forest |

---

## 5.1 k-Nearest Neighbors (kNN) — Complete Theory

### 5.1.1 The Core Idea

kNN is based on one intuition:

> **"Things that are similar belong to the same class."**

Given a new, unlabeled point:
1. Calculate the distance to all training points
2. Find the k nearest neighbors
3. Take a majority vote among those k neighbors
4. Assign the majority class to the new point

kNN is a **lazy learner** — it doesn't build a model during training. It just memorizes all training data and does all computation at prediction time.

### 5.1.2 Distance Metrics — The Mathematics

**Euclidean Distance** (default, like straight-line distance):
```
d(p, q) = √[Σᵢ (pᵢ - qᵢ)²]

For 2D: d = √[(x₁-x₂)² + (y₁-y₂)²]
```

**Manhattan Distance** (like city blocks — only horizontal/vertical moves):
```
d(p, q) = Σᵢ |pᵢ - qᵢ|

For 2D: d = |x₁-x₂| + |y₁-y₂|
```

**Minkowski Distance** (generalization of both):
```
d(p, q) = [Σᵢ |pᵢ - qᵢ|ᵖ]^(1/p)

p=1 → Manhattan
p=2 → Euclidean
```

### 5.1.3 Worked Example (by hand)

Training data (2 features: age, salary):
```
Point | Age | Salary | Class
  A   |  25 |  30000 | Junior
  B   |  40 |  60000 | Senior
  C   |  35 |  55000 | Senior
  D   |  22 |  25000 | Junior
```

New point to classify: Age=30, Salary=40000, k=3

Euclidean distances (after normalizing — always normalize!):
```
d(new, A) = √[(30-25)² + (40000-30000)²] = √[25 + 100000000] ≈ 10000.001
d(new, B) = √[(30-40)² + (40000-60000)²] = √[100 + 400000000] ≈ 20000.003
d(new, C) = √[(30-35)² + (40000-55000)²] = √[25 + 225000000] ≈ 15000.001
d(new, D) = √[(30-22)² + (40000-25000)²] = √[64 + 225000000] ≈ 15000.002
```

3 Nearest: A (Junior), C (Senior), D (Junior)
Vote: Junior=2, Senior=1 → **Predict: Junior**

> **Note:** This example shows why scaling matters! The salary difference (tens of thousands) completely dominates the age difference. After StandardScaler, both features have equal weight.

### 5.1.4 Choosing k

```
k=1: Very sensitive to noise → overfitting
k=large: Too general → underfitting
k=√n: Good starting rule of thumb
```

Always try **odd values of k** for binary classification to avoid ties (2-2 vote impossible with k=3 or 5).

Use cross-validation to find the best k:

```python
from sklearn.model_selection import cross_val_score
import matplotlib.pyplot as plt

k_range = range(1, 31, 2)  # odd values: 1, 3, 5, ..., 29
cv_scores = []

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train, y_train, cv=5, scoring='accuracy')
    cv_scores.append(scores.mean())

plt.plot(k_range, cv_scores)
plt.xlabel('k')
plt.ylabel('CV Accuracy')
plt.title('Finding optimal k')
best_k = k_range[cv_scores.index(max(cv_scores))]
print(f"Best k: {best_k}")
```

### 5.1.5 Complete sklearn Implementation

```python
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, 
                              classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv("diabetes.csv")
print(df.head())
print(df.shape)
print(df['Outcome'].value_counts())  # check class balance

# Features and target
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale (CRITICAL for kNN)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Train kNN
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn.fit(X_train_scaled, y_train)

# Predict
y_pred = knn.predict(X_test_scaled)

# Evaluate
print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — kNN')
plt.show()

# Manual metrics from confusion matrix
TN, FP, FN, TP = cm.ravel()
sensitivity  = TP / (TP + FN)
specificity  = TN / (TN + FP)
precision    = TP / (TP + FP)
accuracy     = (TP + TN) / (TP + TN + FP + FN)
error_rate   = (FP + FN) / (TP + TN + FP + FN)

print(f"\nTP={TP}, TN={TN}, FP={FP}, FN={FN}")
print(f"Sensitivity (Recall) : {sensitivity:.4f}")
print(f"Specificity          : {specificity:.4f}")
print(f"Precision            : {precision:.4f}")
print(f"Accuracy             : {accuracy:.4f}")
print(f"Error Rate           : {error_rate:.4f}")
```

### 5.1.6 kNN for Regression

kNN can also do regression — instead of majority vote, it takes the **average** of the k neighbors' values.

```python
from sklearn.neighbors import KNeighborsRegressor

knn_reg = KNeighborsRegressor(n_neighbors=5)
knn_reg.fit(X_train, y_train)
y_pred = knn_reg.predict(X_test)
```

---

## 5.2 Decision Tree — Complete Theory

### 5.2.1 The Core Idea

A Decision Tree builds a flowchart of questions. At each step, it asks the question that gives the most information — that best splits the data into pure groups.

```
                 [Outlook?]
                /    |    \
           Sunny   Overcast  Rainy
            |        |        |
         [Humidity?] Yes   [Wind?]
          /    \           /    \
        High  Normal    Strong  Weak
         |      |         |      |
         No    Yes        No    Yes
```

### 5.2.2 Entropy — The Mathematical Foundation

**Entropy** measures the impurity or uncertainty in a set of labels.

```
H(S) = -Σₖ p(k) · log₂(p(k))
```

Where p(k) = proportion of class k in set S.

**Intuition:**
- Pure node (all one class): H = 0 (no uncertainty)
- Perfectly mixed node (50/50): H = 1 (maximum uncertainty)
- General formula for binary case:

```
H = -p · log₂(p) - (1-p) · log₂(1-p)
```

**Worked Example:**

Set S has 10 points: 6 Yes, 4 No
```
p(Yes) = 6/10 = 0.6
p(No)  = 4/10 = 0.4

H(S) = -(0.6)·log₂(0.6) - (0.4)·log₂(0.4)
     = -(0.6)·(-0.737) - (0.4)·(-1.322)
     = 0.442 + 0.529
     = 0.971
```

### 5.2.3 Information Gain

**Information Gain (IG)** measures how much entropy is reduced by splitting on a feature.

```
IG(S, A) = H(S) - Σᵥ [|Sᵥ|/|S|] · H(Sᵥ)
```

Where:
- `S` = current dataset
- `A` = feature we are considering splitting on
- `Sᵥ` = subset of S where feature A = value v
- `|Sᵥ|/|S|` = weight (proportion of examples with that value)

**We always choose the feature with the highest Information Gain.**

### 5.2.4 Gini Impurity (Alternative to Entropy)

```
Gini(S) = 1 - Σₖ p(k)²
```

For binary case:
```
Gini = 1 - p² - (1-p)²  = 2p(1-p)
```

- Gini = 0 → pure node
- Gini = 0.5 → maximum impurity (50/50 split)

Gini is slightly faster to compute than Entropy (no logarithm). Results are almost always the same. sklearn defaults to Gini.

### 5.2.5 How a Tree is Built — ID3/CART Algorithm

```
BuildTree(S):
  If S is pure (all same class):
    Return leaf node with that class
  
  For each feature A:
    Calculate IG(S, A)
  
  Best_feature = argmax IG(S, A)
  
  Split S into subsets using Best_feature
  
  For each subset Sᵥ:
    child = BuildTree(Sᵥ)
    Add child to tree
  
  Return tree
```

### 5.2.6 Preventing Overfitting — Pruning

A fully grown tree memorizes every training point. We control tree complexity with:

| Parameter | Effect |
|-----------|--------|
| `max_depth` | Maximum depth of tree. Start with 3-5. |
| `min_samples_split` | Minimum samples required to split a node |
| `min_samples_leaf` | Minimum samples required in a leaf node |
| `max_features` | Number of features to consider at each split |

```python
dt = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42
)
```

### 5.2.7 Complete sklearn Implementation

```python
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

# Load data (PlayTennis example)
df = pd.read_csv("PlayTennis.csv")
print(df.head())

# Encode categorical features
le = LabelEncoder()
for col in df.columns:
    df[col] = le.fit_transform(df[col])

X = df.drop('Play', axis=1)
y = df['Play']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=10
)

# Train Decision Tree
dt = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
dt.fit(X_train, y_train)

# Predict and evaluate
y_pred = dt.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Visualize tree
plt.figure(figsize=(15, 8))
plot_tree(dt, 
          feature_names=X.columns.tolist(),
          class_names=dt.classes_.astype(str).tolist(),
          filled=True,
          rounded=True,
          fontsize=12)
plt.title("Decision Tree (criterion=entropy)")
plt.tight_layout()
plt.show()

# Text representation
print("\nTree structure:")
print(export_text(dt, feature_names=X.columns.tolist()))

# Feature importance
fi = pd.Series(dt.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nFeature Importances:")
print(fi)
fi.plot(kind='bar')
plt.title("Feature Importances")
plt.tight_layout()
plt.show()
```

---

## 5.3 Random Forest — Complete Theory

### 5.3.1 Why Random Forest?

Single Decision Trees have high variance — small changes in data can produce very different trees. Random Forest fixes this through **ensemble learning**.

> **Wisdom of the crowd:** 100 independent, somewhat-random trees, each making a different error, will collectively make fewer errors than any single tree.

### 5.3.2 The Algorithm — Bagging + Random Subspace

Random Forest combines two ideas:

**1. Bootstrap Aggregating (Bagging):**
- For each tree, sample the training data **with replacement** (bootstrap sample)
- A bootstrap sample is the same size as the original dataset
- ~63% of original points appear (some appear multiple times, ~37% are never picked = "out-of-bag" samples)

**2. Random Subspace Method:**
- At each split, only consider a **random subset of features**
- Typical: `max_features = √(total_features)` for classification
- This decorrelates the trees — they make different errors

**3. Aggregation:**
- Classification: **majority vote** across all trees
- Regression: **average** of all tree outputs

```
                Training Data (n samples)
               /          |           \
        Bootstrap 1   Bootstrap 2   Bootstrap N
             |              |             |
          Tree 1         Tree 2        Tree N
         (rand feats)  (rand feats)  (rand feats)
               \             |            /
                \            |           /
                 → MAJORITY VOTE → Final Prediction
```

### 5.3.3 Why This Reduces Overfitting

- Each tree overfits to its bootstrap sample
- But different trees overfit in different directions (they're built on different data with different features)
- Errors are random and **uncorrelated**
- When you average uncorrelated errors, they cancel out → variance drops dramatically
- Bias stays roughly the same as a single tree

This is the core magic of ensemble methods: **reducing variance without increasing bias.**

### 5.3.4 Out-of-Bag (OOB) Error

The ~37% of data not used to train each tree = out-of-bag samples. We can evaluate each tree on its OOB samples to get a free validation estimate — no need for a separate validation set!

```python
rf = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=42)
rf.fit(X_train, y_train)
print(f"OOB Score: {rf.oob_score_:.4f}")
```

### 5.3.5 Feature Importance in Random Forest

Random Forest gives us a powerful measure of feature importance:

```
Importance of feature A = average decrease in impurity (Gini/entropy) 
                          caused by splits on A, averaged across all trees
```

Higher importance = feature is more useful for prediction.

### 5.3.6 Complete sklearn Implementation

```python
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, confusion_matrix, 
                              classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv("diabetes.csv")
X = df.drop('Outcome', axis=1)
y = df['Outcome']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=100,      # number of trees
    criterion='gini',      # 'gini' or 'entropy'
    max_depth=None,        # None = fully grown (then bagging reduces variance)
    max_features='sqrt',   # √(n_features) features per split
    bootstrap=True,        # use bootstrap sampling
    oob_score=True,        # compute OOB score
    random_state=42,
    n_jobs=-1              # use all CPU cores (faster training)
)
rf.fit(X_train, y_train)

# Predict
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]  # probability of class 1

# Evaluate
print(f"Train Accuracy : {accuracy_score(y_train, rf.predict(X_train)):.4f}")
print(f"Test Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"OOB Score      : {rf.oob_score_:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Feature importance plot
fi = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
fi.plot(kind='barh', figsize=(8, 5), color='steelblue')
plt.xlabel('Importance')
plt.title('Feature Importance — Random Forest')
plt.tight_layout()
plt.show()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — Random Forest')
plt.show()
```

---

## 5.4 Support Vector Machine (SVM) — Complete Theory

### 5.4.1 The Core Idea

SVM finds the **optimal hyperplane** that separates two classes with the **maximum margin**.

```
              Class B  ●  ●
                      ●      
    ───────────────────────────── ← decision boundary (hyperplane)
    
    margin ←→ margin

     ○     ○
      ○  ○    Class A
```

The **margin** = distance from the decision boundary to the closest points of each class.

The **support vectors** = the data points on the margin boundary (closest points to the boundary). These are the ONLY points that define the boundary — remove all other points and the model doesn't change.

### 5.4.2 Mathematical Formulation

For a linear SVM with features x and labels y ∈ {-1, +1}:

The decision boundary is a hyperplane:
```
w · x + b = 0
```

Where:
- `w` = weight vector (normal to the hyperplane)
- `b` = bias term
- `·` = dot product

The margin = `2 / ||w||`

**Hard Margin SVM** (linearly separable data):
```
Minimize:    (1/2) ||w||²          (maximize margin)
Subject to:  yᵢ(w·xᵢ + b) ≥ 1    for all i
```

**Soft Margin SVM** (with misclassifications allowed):
```
Minimize:    (1/2)||w||² + C·Σ ξᵢ
Subject to:  yᵢ(w·xᵢ + b) ≥ 1 - ξᵢ  and  ξᵢ ≥ 0
```

Where:
- `ξᵢ` (xi) = slack variable — how far misclassified point is on the wrong side
- `C` = regularization parameter — controls the tradeoff

**The C parameter:**
```
Large C → penalty for misclassification is high → small margin → tries hard to classify everything → overfits
Small C → allows more misclassifications → wider margin → underfits
```

### 5.4.3 The Kernel Trick — Handling Non-Linear Data

When data is not linearly separable in the original feature space, we map it to a **higher-dimensional space** where it becomes separable.

```
2D (not separable) → 3D (separable with a plane)
```

The **kernel function** computes the dot product in higher-dimensional space without actually computing the transformation:

```
K(x, z) = φ(x) · φ(z)
```

**Common kernels:**

| Kernel | Formula | Use case |
|--------|---------|---------|
| **Linear** | K(x,z) = x·z | Linearly separable, high-dimensional data (text) |
| **RBF/Gaussian** | K(x,z) = exp(-γ||x-z||²) | Default. Works for most non-linear problems |
| **Polynomial** | K(x,z) = (x·z + c)^d | Polynomial relationships |
| **Sigmoid** | K(x,z) = tanh(αx·z + c) | Rarely used |

**The γ (gamma) parameter in RBF:**
```
Large γ → each training point has narrow influence → complex boundary → overfits
Small γ → each training point has wide influence → smooth boundary → underfits
```

### 5.4.4 SVM for Multi-Class (beyond binary)

SVM is natively binary. For multi-class, sklearn uses:
- **One-vs-One (OvO):** Train C(C-1)/2 classifiers for C classes; majority vote
- **One-vs-Rest (OvR):** Train C classifiers, each separates one class from the rest

sklearn's SVC uses OvO by default.

### 5.4.5 Complete sklearn Implementation

```python
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, 
                              classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv("cancer.csv")
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']   # M = Malignant, B = Benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Scale features (CRITICAL for SVM)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Train SVM
svm = SVC(
    C=1.0,           # regularization
    kernel='rbf',    # 'linear', 'rbf', 'poly', 'sigmoid'
    gamma='scale',   # 'scale', 'auto', or float
    random_state=42,
    probability=True  # enables predict_proba
)
svm.fit(X_train_scaled, y_train)

# Predict
y_pred = svm.predict(X_test_scaled)

# Evaluate
print(f"Train Accuracy: {accuracy_score(y_train, svm.predict(X_train_scaled)):.4f}")
print(f"Test Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Benign', 'Malignant'],
            yticklabels=['Benign', 'Malignant'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix — SVM')
plt.show()

# Try different kernels
for kernel in ['linear', 'rbf', 'poly']:
    svm_k = SVC(kernel=kernel, C=1.0, random_state=42)
    svm_k.fit(X_train_scaled, y_train)
    acc = accuracy_score(y_test, svm_k.predict(X_test_scaled))
    print(f"Kernel={kernel:8s}: Accuracy = {acc:.4f}")
```

---

## 5.5 Confusion Matrix — Complete Theory

### 5.5.1 The Four Outcomes

For a **binary classifier** (Positive = 1, Negative = 0):

```
                     PREDICTED
                 Positive   Negative
ACTUAL Positive |   TP    |   FN   |
       Negative |   FP    |   TN   |
```

- **TP (True Positive):** Model said positive, actually positive → Correct!
- **TN (True Negative):** Model said negative, actually negative → Correct!
- **FP (False Positive / Type I Error):** Model said positive, actually negative → Wrong!
- **FN (False Negative / Type II Error):** Model said negative, actually positive → Wrong!

**Medical analogy:**
```
Disease positive:
  TP = correctly diagnosed sick person
  TN = correctly cleared healthy person
  FP = healthy person told they're sick (unnecessary treatment/anxiety)
  FN = sick person told they're healthy (dangerous! missed diagnosis)
```

### 5.5.2 All Evaluation Metrics — Formulas and Meaning

| Metric | Formula | Meaning | When critical |
|--------|---------|---------|---------------|
| **Accuracy** | (TP+TN)/(TP+TN+FP+FN) | Overall correctness | Balanced datasets |
| **Error Rate** | (FP+FN)/(TP+TN+FP+FN) | Overall mistakes | = 1 - Accuracy |
| **Sensitivity (Recall, TPR)** | TP/(TP+FN) | Detected % of actual positives | Medical, fraud detection |
| **Specificity (TNR)** | TN/(TN+FP) | Detected % of actual negatives | Spam filters |
| **Precision (PPV)** | TP/(TP+FP) | Accuracy when we say "positive" | When FP is costly |
| **F1 Score** | 2·(Precision·Recall)/(Precision+Recall) | Harmonic mean of precision/recall | Imbalanced datasets |
| **False Positive Rate** | FP/(FP+TN) | = 1 - Specificity | ROC curves |

### 5.5.3 The Precision-Recall Tradeoff

There is always a tradeoff between precision and recall. You can't maximize both.

```
Example: Cancer detection
- Lower decision threshold → more people classified as cancer
  → FN decreases (miss fewer real cases) → Recall increases
  → FP increases (more false alarms) → Precision decreases

- Higher decision threshold → fewer people classified as cancer
  → FP decreases → Precision increases
  → FN increases → Recall decreases
```

**F1 Score** balances both:
```
F1 = 2 × (Precision × Recall) / (Precision + Recall)
   = 2TP / (2TP + FP + FN)
```

### 5.5.4 Worked Example

A model makes 100 predictions for disease detection:

```
Actual Positive: 60 patients truly sick
Actual Negative: 40 patients truly healthy

Confusion Matrix:
                Pred Positive  Pred Negative
Actual Positive      50              10
Actual Negative       5              35

TP=50, FN=10, FP=5, TN=35
```

Calculations:
```
Accuracy    = (50+35)/100 = 0.85 = 85%
Error Rate  = (5+10)/100  = 0.15 = 15%
Sensitivity = 50/(50+10)  = 0.833 = 83.3%  ← 16.7% of sick people missed!
Specificity = 35/(35+5)   = 0.875 = 87.5%
Precision   = 50/(50+5)   = 0.909 = 90.9%
F1 Score    = 2(0.909×0.833)/(0.909+0.833) = 0.870
```

### 5.5.5 Multi-Class Confusion Matrix

For 3 classes (e.g., setosa, versicolor, virginica):

```
                  Predicted
              Set  Ver  Vir
Actual  Set  [ 19   0    0 ]
        Ver  [  0  17    2 ]
        Vir  [  0   1   21 ]
```

Each row = actual class, each column = predicted class.
Diagonal = correct predictions.
Off-diagonal = misclassifications.

### 5.5.6 Complete Evaluation Code

```python
from sklearn.metrics import (confusion_matrix, classification_report, 
                              accuracy_score, precision_score, 
                              recall_score, f1_score)
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

def full_evaluation(y_test, y_pred, classes=None, title="Model"):
    """Complete evaluation function — use for any classifier."""
    
    print(f"\n{'='*50}")
    print(f"  {title} — Performance Report")
    print(f"{'='*50}")
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    # Binary case: extract TP, TN, FP, FN
    if len(cm) == 2:
        TN, FP, FN, TP = cm.ravel()
        print(f"\nConfusion Matrix Values:")
        print(f"  TP (True Positives)  = {TP}")
        print(f"  TN (True Negatives)  = {TN}")
        print(f"  FP (False Positives) = {FP}")
        print(f"  FN (False Negatives) = {FN}")
        
        accuracy    = (TP+TN)/(TP+TN+FP+FN)
        error_rate  = (FP+FN)/(TP+TN+FP+FN)
        sensitivity = TP/(TP+FN)
        specificity = TN/(TN+FP)
        precision   = TP/(TP+FP)
        f1          = 2*(precision*sensitivity)/(precision+sensitivity)
        
        print(f"\nMetrics:")
        print(f"  Accuracy    = {accuracy:.4f}  ({accuracy*100:.1f}%)")
        print(f"  Error Rate  = {error_rate:.4f}  ({error_rate*100:.1f}%)")
        print(f"  Sensitivity = {sensitivity:.4f}  (Recall / TPR)")
        print(f"  Specificity = {specificity:.4f}  (TNR)")
        print(f"  Precision   = {precision:.4f}")
        print(f"  F1 Score    = {f1:.4f}")
    
    # Full classification report
    print(f"\nFull Classification Report:")
    if classes:
        print(classification_report(y_test, y_pred, target_names=classes))
    else:
        print(classification_report(y_test, y_pred))
    
    # Plot confusion matrix
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes or np.unique(y_test),
                yticklabels=classes or np.unique(y_test))
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix — {title}')
    plt.tight_layout()
    plt.show()

# Usage:
# full_evaluation(y_test, y_pred, classes=['No Disease', 'Disease'], title='kNN')
# full_evaluation(y_test, y_pred, classes=['Benign', 'Malignant'], title='SVM')
```

---

## 5.6 Algorithm Comparison — When to Use What

| Algorithm | Best For | Weaknesses | Needs Scaling | Interpretable |
|-----------|----------|------------|---------------|---------------|
| **kNN** | Small datasets, non-linear boundaries | Slow predict, sensitive to scale | ✅ Yes | Moderate |
| **Decision Tree** | Interpretability, mixed data types | Overfits easily | ❌ No | ✅ Yes |
| **Random Forest** | General purpose, good defaults | Slower than single tree, hard to interpret | ❌ No | ❌ No |
| **SVM (RBF)** | High-dimensional, clear margin | Slow on large data, needs tuning | ✅ Yes | ❌ No |
| **SVM (Linear)** | Text classification, high dimensions | Linear boundary only | ✅ Yes | Partial |

---

## 5.7 🌟 Extra: Hyperparameter Tuning (Industry Essential)

Every algorithm has **hyperparameters** — settings we choose before training. Finding the best combination is called **hyperparameter tuning**.

### GridSearchCV — Exhaustive Search

```python
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

# Define parameter grid
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto', 0.1, 0.01]
}

# Grid search with cross-validation
grid = GridSearchCV(
    SVC(),
    param_grid,
    cv=5,               # 5-fold cross-validation
    scoring='accuracy',
    n_jobs=-1,          # use all CPU cores
    verbose=1           # print progress
)
grid.fit(X_train_scaled, y_train)

print(f"Best parameters: {grid.best_params_}")
print(f"Best CV score  : {grid.best_score_:.4f}")

# Use best model
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test_scaled)
```

### RandomizedSearchCV — Faster for Large Grids

```python
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators': randint(50, 500),
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': randint(2, 20)
}

rand_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_dist,
    n_iter=50,    # try 50 random combinations (not all)
    cv=5,
    scoring='accuracy',
    random_state=42
)
rand_search.fit(X_train, y_train)
print(f"Best params: {rand_search.best_params_}")
```

---

## 5.8 🌟 Extra: Handling Imbalanced Datasets

Real-world datasets are often imbalanced (e.g., 95% no-fraud, 5% fraud).

**Problem:** A model that always predicts "no fraud" gets 95% accuracy — but is useless!

**Solutions:**

```python
# 1. Use stratify in train_test_split (always do this)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Use class_weight='balanced' in model
rf = RandomForestClassifier(class_weight='balanced', random_state=42)
svm = SVC(class_weight='balanced')
dt  = DecisionTreeClassifier(class_weight='balanced')

# 3. Use F1 score / precision / recall instead of accuracy for evaluation
# 4. Oversampling (SMOTE) — advanced technique
from imblearn.over_sampling import SMOTE
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train, y_train)
```

---

## 5.9 🌟 Extra: The ML Pipeline — Industry Standard

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# Full pipeline: preprocess → reduce dims → model
pipe = Pipeline([
    ('scaler', StandardScaler()),       # step 1: scale
    ('pca', PCA(n_components=10)),      # step 2: reduce dimensions (optional)
    ('svm', SVC(random_state=42))       # step 3: model
])

# GridSearch over pipeline parameters (use __ notation)
param_grid = {
    'svm__C':      [0.1, 1, 10],
    'svm__kernel': ['rbf', 'linear'],
    'pca__n_components': [5, 10, 15]
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)
print(grid.best_params_)
```

Pipelines automatically handle data leakage — when you call `grid.fit(X_train, y_train)`, each fold's preprocessing is fit ONLY on that fold's training data.

---

## Summary: Complete sklearn Cheatsheet

### Regression
```python
# Linear Regression
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit(X_train, y_train)
print(model.coef_, model.intercept_)
y_pred = model.predict(X_test)

# Polynomial Regression
from sklearn.preprocessing import PolynomialFeatures
poly = PolynomialFeatures(degree=3)
X_poly = poly.fit_transform(X)

# Evaluation
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
r2_score(y_test, y_pred)
mean_squared_error(y_test, y_pred)
mean_absolute_error(y_test, y_pred)
```

### Classification
```python
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

knn = KNeighborsClassifier(n_neighbors=5)
dt  = DecisionTreeClassifier(criterion='entropy', max_depth=5)
rf  = RandomForestClassifier(n_estimators=100, random_state=42)
svm = SVC(C=1.0, kernel='rbf')

# All follow same API:
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Evaluation
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
accuracy_score(y_test, y_pred)
confusion_matrix(y_test, y_pred)
classification_report(y_test, y_pred)
```

### Preprocessing
```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

df = pd.get_dummies(df, drop_first=True)        # one-hot encode
df['col'].fillna(df['col'].mean(), inplace=True) # fill missing
```

---

## Exam Quick Reference

### Must-Know Formulas

```
R²          = 1 - SS_res/SS_tot
MSE         = (1/n)·Σ(y - ŷ)²
RMSE        = √MSE
MAE         = (1/n)·Σ|y - ŷ|
Entropy     = -Σ p(k)·log₂(p(k))
Info Gain   = H(S) - Σ |Sᵥ|/|S| · H(Sᵥ)
Gini        = 1 - Σ p(k)²
Accuracy    = (TP+TN) / (TP+TN+FP+FN)
Error Rate  = (FP+FN) / (TP+TN+FP+FN) = 1 - Accuracy
Sensitivity = TP / (TP+FN)
Specificity = TN / (TN+FP)
Precision   = TP / (TP+FP)
F1 Score    = 2·(Precision·Recall)/(Precision+Recall)
```

### train_test_split output order
```
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
```

### Key sklearn classes
```
LinearRegression          → sklearn.linear_model
PolynomialFeatures        → sklearn.preprocessing
train_test_split          → sklearn.model_selection
KNeighborsClassifier      → sklearn.neighbors
DecisionTreeClassifier    → sklearn.tree
RandomForestClassifier    → sklearn.ensemble
SVC                       → sklearn.svm
confusion_matrix          → sklearn.metrics
classification_report     → sklearn.metrics
StandardScaler            → sklearn.preprocessing
```

---

*Notes prepared for LJ Institute of Engineering & Technology, Sem IV, 2026*
*Subject: Fundamentals of Computer Science using Python – II*
*Units covered: 4 (Regression) and 5 (Classification)*